# Pipeline de Extracción de Contaminantes de la EEA

Este *notebook* implementa un flujo completo de extracción, transformación y carga (ETL) para obtener datos históricos de la *European Environment Agency* (EEA). El objetivo es consolidar una base de datos sobre mediciones de contaminantes atmosféricos de 12 ciudades españolas.

La API de la EEA permite acceder a series temporales sobre diferentes contaminantes atmosféricos de diversas ciudades. En cada ciudad de las que se disponen datos, se encuentran diferentes dispositivos que miden distintos contaminantes, por lo que para cada ciudad se dispone de datos sobre diferentes contaminantes.

En este proyecto, hemos optado por:

* Formato *parquet*, ya que es un formato de almacenamiento columnar que ofrece una compresión superior y un rendimiento de lectura mucho más rápido que el CSV tradicional. Además, es el formato en que la API devuelve los archivos.

* Granularidad: seleccionamos datos con agregación horaria, permitiendo análisis detallados de ciclos diarios.

* Alcance: tomamos los datos desde 2013 hasta 2024.

Por tanto, seguimos 4 fases para realizar este proceso ETL:

1. **Identificación de contaminantes (Pollutants)**: realizamos una petición a la API para obtener el catálogo de contaminantes monitorizados (como $NO_2$, $PM_{10}$, $O_3$, etc.). Esto asegura que el script sea compatible con cualquier nueva sustancia que la agencia comience a reportar en el futuro.

2. **Generación de consultas (*Payloads*)**: debido al gran volumen de información, la API no entrega los archivos directamente. Entonces, construimos solicitudes de red complejas (*JSON payloads*) que combinan: geolocalización (país + ciudad), parámetros técnicos (tipo de fuente y método de agregación) y un rango temporal. La API responde con una lista de URLs temporales para la descarga de los fragmentos de datos.

3. **Descarga optimizada y gestión de memoria**: para evitar el colapso de la memoria RAM al manejar cientos de archivos, implementamos una descarga por flujo de datos (*streaming*): los archivos se descargan en bloques de 8KB, se organiza el sistema de archivos en una jerarquía de carpetas por país ISO y se implementa un sistema de reintentos y verificación de existencia para evitar descargas duplicadas.

4. **Validación de esquemas y normalización final**: una vez en el disco, el script realiza una inspección profunda de cada archivo: verificamos qué contaminante contiene realmente cada archivo (detectando posibles inconsistencias en la API). Además, renombramos los archivos para que tengan una nomenclatura estándar: CIUDAD-CONTAMINANTE_INDICE.parquet, facilitando su posterior uso. Posteriormente, se realizan otras transformaciones de formato a los datos.

In [ ]:
# ==============================================================================
# IMPORTACIÓN DE LIBRERÍAS Y DEPENDENCIAS
# ==============================================================================

from aiohttp import request           # Estándar para realizar peticiones HTTP (GET/POST) a la API de la EEA.
import requests
import os                             # Funciones básicas del sistema operativo.
from pathlib import Path              # Interfaz orientada a objetos para rutas (recomendado sobre os.path).
import pandas as pd                   # Motor principal para lectura de archivos Parquet y validación de columnas.
from tqdm import tqdm                 # Visualización de barras de progreso.
from collections import defaultdict   # Diccionarios con inicialización automática.

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# ==============================================================================
# CONFIGURACIÓN DE LA API - European Environment Agency (EEA)
# ==============================================================================

# Esta URL es la base para el servicio de descarga de datos de la EEA.
# Se utiliza como prefijo para todas las peticiones a la API.
BASE_URL = "https://eeadmz1-downloads-api-appservice.azurewebsites.net"

# ENDPOINTS DISPONIBLES:
# ----------------------
# /Country:           Obtiene el listado de países disponibles.
# /City:              Obtiene el listado de ciudades filtradas por país.
# /Pollutant:         Lista de contaminantes atmosféricos monitoreados.
# /ParquetFile/urls:  Punto de acceso principal para obtener las rutas de descarga de los archivos de datos en formato Parquet.

In [3]:
# ==============================================================================
# CONFIGURACIÓN ESPECÍFICA PARA ESPAÑA Y CIUDADES SELECCIONADAS
# ==============================================================================

# Lista de códigos de país que se van a consultar en la API.
# "ES" corresponde al código ISO de España.
country_codes = ["ES"]

# Lista de ciudades objetivo que queremos analizar.
# Solo se procesarán datos de estas ciudades concretas si aparecen en la respuesta de la API.
target_cities = ["Madrid", "Barcelona", "Bilbao", "Sevilla", 
                 "Valencia", "Valladolid", "Murcia", "Santa Cruz de Tenerife",
                 "A Coruña", "Zaragoza", "Alicante/Alacant", "Albacete"]

# Diccionario donde se almacenarán las ciudades encontradas por país.
# La estructura será algo como:
# {
#     "ES": ["Madrid", "Barcelona", ...]
# }
all_cities = {}

# Mensaje informativo para indicar que comienza el proceso de filtrado.
print("Filtrando ciudades específicas para España...")

# Bucle que recorre todos los códigos de país definidos anteriormente.
# En este caso solo hay uno ("ES"), pero el código está preparado para varios países.
for cc in tqdm(country_codes):

    # Se realiza una petición POST a la API para obtener la lista de ciudades
    # disponibles para el país especificado.
    # BASE_URL debe haberse definido previamente.
    # Se envía el código del país dentro de una lista como cuerpo JSON.
    r = requests.post(f"{BASE_URL}/City", json=[cc])

    # Se convierte la respuesta de la API (en formato JSON) a un objeto Python.
    # Normalmente será una lista de diccionarios con información de cada ciudad.
    city_list = r.json()

    # Se filtran las ciudades recibidas para quedarse únicamente con las que
    # aparecen en la lista target_cities.
    # Cada elemento de city_list es un diccionario con información de la ciudad,
    # y se accede al nombre mediante la clave "cityName".
    all_cities[cc] = [
        ci["cityName"] for ci in city_list 
        if ci["cityName"] in target_cities
    ]

# Se obtienen las ciudades encontradas para España dentro del diccionario.
# Si no existiera la clave "ES", devolvería una lista vacía.
found_cities = all_cities.get("ES", [])

# Se imprime un resumen indicando cuántas ciudades objetivo se han encontrado
# realmente en la respuesta de la API frente al total de ciudades buscadas.
print(f"Ciudades encontradas: {len(found_cities)} de {len(target_cities)}")

Filtrando ciudades específicas para España...


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  1.79it/s]

Ciudades encontradas: 12 de 12


In [4]:
# ==============================================================================
# OBTENCIÓN DE CONTAMINANTES (POLLUTANTS)
# ==============================================================================

# Mensaje informativo que indica que se va a solicitar a la API
# la lista de contaminantes disponibles en el sistema.
print("Fetching pollutants list...")

# Se realiza una petición HTTP GET al endpoint "/Pollutant" de la API.
# Este endpoint devuelve información sobre los contaminantes que la API
# puede proporcionar (por ejemplo: NO2, O3, PM10, PM2.5, etc.).
# BASE_URL debe haberse definido previamente con la URL base de la API.
r = requests.get(f"{BASE_URL}/Pollutant")

# Se convierte la respuesta recibida (en formato JSON) a un objeto Python.
# Normalmente será una lista de diccionarios, donde cada diccionario
# contiene información sobre un contaminante (nombre, código, descripción, etc.).
pollutants = r.json()

# Se extrae únicamente el código o notación de cada contaminante.
# "notation" suele ser el identificador estándar del contaminante
# (por ejemplo: "NO2", "O3", "PM10", "PM2.5").
# Se utiliza una list comprehension para recorrer la lista de contaminantes
# y crear una nueva lista solo con esos códigos.
pollutant_codes = [p["notation"] for p in pollutants]

# Se muestra por pantalla el número total de contaminantes encontrados.
# Esto permite comprobar rápidamente que la API ha respondido correctamente
# y cuántos contaminantes están disponibles para futuras consultas.
print(f"Found {len(pollutant_codes)} pollutants.")

Fetching pollutants list...
Found 648 pollutants.


In [5]:
# ==============================================================================
# GESTIÓN DEL SISTEMA DE ARCHIVOS
# ==============================================================================

# Se define la ruta base donde se almacenarán los datasets descargados.
# En este caso, usamos una ruta relativa para que la carpeta "datasets"
# se cree un nivel por encima de la carpeta actual del proyecto.
#
# Si el proyecto está en:
# TFM/TFM_github/
#
# entonces los datos se guardarán en:
# TFM/datasets/
BASE_PATH = Path("..", "..") / "datasets"

# Se define la carpeta donde se guardarán los datos en formato Parquet.
# En este caso se crea la ruta:
# ../datasets/eea_parquet
DATA_DIR = BASE_PATH / "eea_parquet"

# Se crea la carpeta en el sistema de archivos si todavía no existe.
# Parámetros utilizados:
# - parents=True  -> crea automáticamente las carpetas intermedias si no existen
# - exist_ok=True -> evita que el programa genere un error si la carpeta ya existe
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Carpeta de datos preparada en: {DATA_DIR}")

📂 Carpeta de datos preparada en: ..\..\datasets\eea_parquet


In [ ]:
# ==============================================================================
# GENERACIÓN DE ENLACES DE DESCARGA
# ==============================================================================

# Lista donde se almacenará la información de todos los archivos disponibles
# para descargar. Cada elemento será un diccionario con:
# - URL del archivo
# - país
# - ciudad
# - contaminante
all_files = []

# Email requerido por la API para generar enlaces de descarga.
# Algunas APIs lo usan para control de uso o para enviar notificaciones.
email = "email@gmail.com" 

# Se obtienen las ciudades encontradas previamente para España.
# Si por alguna razón la clave "ES" no existe en el diccionario all_cities,
# se devolverá una lista vacía para evitar errores.
lista_ciudades_esp = all_cities.get("ES", [])

# Bucle principal que recorre todas las ciudades seleccionadas de España.
# tqdm se utiliza para mostrar una barra de progreso durante la ejecución,
# lo cual es útil cuando el proceso puede tardar bastante tiempo.
for city in tqdm(lista_ciudades_esp, desc="Procesando ciudades de España"):

    # Se define explícitamente el código del país.
    # En este caso siempre será "ES", pero se deja así para mantener claridad.
    country_code = "ES"
    
    # Segundo bucle que recorre todos los contaminantes disponibles
    # obtenidos anteriormente desde el endpoint /Pollutant.
    for pollutant in pollutant_codes:

        # Se construye el payload (cuerpo de la petición POST) que se enviará a la API.
        # Este diccionario define exactamente qué datos queremos descargar.
        payload = {
            "countries": [country_code],  # País del que queremos los datos
            "cities": [city],             # Ciudad específica
            "pollutants": [pollutant],    # Contaminante específico
            "dataset": 2,                 # Tipo de dataset (según especificación de la API)
            "source": "EEA",              # Fuente de datos (European Environment Agency)
            "method": "Automatic",        # Método de medición
            "dateTimeStart": "2013-01-01T00:00:00Z",  # Fecha inicial del rango temporal
            "dateTimeEnd": "2024-12-31T23:59:59Z",    # Fecha final del rango temporal
            "aggregationType": "hour",    # Nivel de agregación temporal (datos horarios)
            "email": email,               # Email requerido por la API
            "compress": False             # Indica si el archivo se quiere comprimido o no
        }

        # Se usa un bloque try/except para manejar posibles errores en la petición
        # (problemas de red, timeouts, errores de la API, etc.).
        try:

            # Se envía la petición POST al endpoint que genera las URLs de descarga
            # para los archivos en formato Parquet.
            r = requests.post(f"{BASE_URL}/ParquetFile/urls", json=payload, timeout=30)

            # Algunas respuestas de la API pueden incluir un BOM (Byte Order Mark).
            # lstrip("\ufeff") elimina ese carácter invisible para evitar problemas
            # al procesar el texto.
            text = r.text.lstrip("\ufeff") 

            # La API devuelve las URLs separadas por saltos de línea.
            # Se divide el texto en líneas y se eliminan espacios en blanco.
            # También se filtran líneas vacías.
            urls = [u.strip() for u in text.splitlines() if u.strip()]

            # Se recorren todas las URLs obtenidas.
            # Puede haber más de una URL si los datos están divididos en varios archivos.
            for url in urls:

                # Se añade un diccionario con la información del archivo a la lista all_files.
                all_files.append({
                    "url": url,            # URL de descarga del archivo parquet
                    "country": country_code, # País
                    "city": city,            # Ciudad
                    "pollutant": pollutant   # Contaminante
                })

        # Si ocurre cualquier error durante la petición o el procesamiento
        # de la respuesta, se captura la excepción y se muestra un mensaje
        # indicando qué ciudad y contaminante causaron el problema.
        except Exception as e:
            print(f"Error pidiendo URL para {city}-{pollutant}: {e}")

In [ ]:
# ==============================================================================
# EJECUCIÓN DE DESCARGAS (CON ESTRUCTURA DE SUBDIRECTORIOS POR CIUDAD)
# ==============================================================================

# Se crea un diccionario con valor por defecto entero (0).
# Se utilizará para llevar un contador de archivos descargados por combinación
# ciudad–contaminante, evitando así sobrescribir archivos con el mismo nombre.
counter = defaultdict(int)

# Se recorre la lista all_files generada anteriormente, que contiene:
# - URL de descarga
# - país
# - ciudad
# - contaminante
# tqdm muestra una barra de progreso durante la descarga.
for f in tqdm(all_files, desc="Descargando datos por ciudad"):

    # Se limpia la URL eliminando posibles caracteres invisibles (BOM)
    # y espacios en blanco al principio o al final.
    url = f["url"].lstrip("\ufeff").strip()

    # Si por alguna razón la URL no empieza por "http",
    # se ignora para evitar errores de descarga.
    if not url.startswith("http"): 
        continue

    # --------------------------------------------------------------------------
    # LIMPIEZA DE NOMBRES PARA EVITAR PROBLEMAS EN EL SISTEMA DE ARCHIVOS
    # --------------------------------------------------------------------------

    # Se reemplazan espacios por "_" y "/" por "-"
    # para asegurar que el nombre de la carpeta sea válido en el sistema operativo.
    city_clean = f["city"].replace(" ", "_").replace("/", "-")

    # Se limpia también el nombre del contaminante.
    pollutant_clean = f["pollutant"].replace(" ", "_")

    # Se obtiene el código del país.
    country = f["country"]

    # --------------------------------------------------------------------------
    # CREACIÓN DE LA ESTRUCTURA DE DIRECTORIOS
    # --------------------------------------------------------------------------

    # Se define la estructura de carpetas donde se guardarán los archivos:
    # eea_parquet / ES / Nombre_Ciudad
    #
    # Ejemplo:
    # datasets/
    #   eea_parquet/
    #       ES/
    #           Madrid/
    #           Barcelona/
    #           Valencia/
    city_dir = DATA_DIR / country / city_clean

    # Se crea la carpeta si no existe.
    # parents=True crea toda la jerarquía si es necesario.
    # exist_ok=True evita error si la carpeta ya existe.
    city_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------------------------
    # CONTROL DE DUPLICADOS
    # --------------------------------------------------------------------------

    # Se crea una clave única combinando ciudad y contaminante.
    key = f"{city_clean}_{pollutant_clean}"

    # Se incrementa el contador para esa combinación.
    # Esto permite numerar archivos si existen varios datasets
    # para el mismo contaminante en la misma ciudad.
    counter[key] += 1
    
    # --------------------------------------------------------------------------
    # GENERACIÓN DEL NOMBRE DEL ARCHIVO
    # --------------------------------------------------------------------------

    # Formato del archivo:
    # Ciudad-Contaminante-Indice.parquet
    #
    # Ejemplo:
    # Madrid-NO2-1.parquet
    # Madrid-NO2-2.parquet
    # Barcelona-PM10-1.parquet
    filename = f"{city_clean}-{pollutant_clean}-{counter[key]}.parquet"

    # Ruta completa donde se guardará el archivo.
    path = city_dir / filename

    # Si el archivo ya existe, se omite la descarga
    # para evitar sobrescribir datos ya descargados.
    if path.exists(): 
        continue

    # --------------------------------------------------------------------------
    # DESCARGA DEL ARCHIVO
    # --------------------------------------------------------------------------

    try:
        # Se realiza la petición GET para descargar el archivo.
        # stream=True permite descargar el archivo en bloques
        # en lugar de cargarlo entero en memoria (más eficiente para archivos grandes).
        with requests.get(url, stream=True, timeout=60) as r:

            # Si la respuesta HTTP indica error (4xx o 5xx),
            # se lanzará una excepción automáticamente.
            r.raise_for_status()

            # Se abre el archivo destino en modo binario ("wb").
            with open(path, "wb") as out_file:

                # Se descarga el contenido en fragmentos de 8192 bytes (8 KB).
                # Esto evita consumir demasiada memoria con archivos grandes.
                for chunk in r.iter_content(chunk_size=8192):

                    # Se escribe el fragmento en el archivo si contiene datos.
                    if chunk: 
                        out_file.write(chunk)

    # Si ocurre algún error durante la descarga o escritura del archivo,
    # se captura la excepción y se muestra un mensaje informativo.
    except Exception as e:
        print(f"Error descargando {city_clean} ({pollutant_clean}): {e}")

# Mensaje final indicando que el proceso ha terminado
# y mostrando la ruta donde se han guardado los archivos descargados.
print(f"\nProceso finalizado. Archivos guardados en: {DATA_DIR}")


Proceso finalizado. Archivos guardados en: ..\..\datasets\eea_parquet


In [ ]:
# ==============================================================================
# NORMALIZACIÓN FINAL
# ==============================================================================

# Posibles nombres de columnas donde puede aparecer el contaminante
# dentro de los datasets descargados. Dependiendo de la fuente o versión
# del dataset, la columna puede llamarse de distintas formas.
POLLUTANT_COLUMNS = ["pollutant", "Pollutant", "pollutantNotation", "notation"]

# Diccionario contador para evitar conflictos de nombres cuando varios
# archivos generen el mismo nombre final después de la normalización.
name_counter = defaultdict(int)

# Se recorren todas las carpetas dentro del directorio DATA_DIR.
# Cada carpeta representa un país (por ejemplo: "ES").
# Solo se seleccionan elementos que sean directorios.
for country_folder in tqdm([p for p in DATA_DIR.iterdir() if p.is_dir()], desc="Normalizando"):

    # Dentro de cada carpeta de país se buscan todos los archivos con extensión .parquet
    for parquet_file in country_folder.glob("*.parquet"):

        try:
            # Se carga el archivo parquet en un DataFrame de pandas.
            # Esto permite inspeccionar su estructura y metadatos.
            df = pd.read_parquet(parquet_file)

            # Se intenta encontrar cuál de las posibles columnas de contaminante
            # está presente en el DataFrame.
            # next(...) devuelve la primera coincidencia encontrada o None si no existe.
            pollutant_col = next((col for col in POLLUTANT_COLUMNS if col in df.columns), None)
            
            # Si se ha encontrado una columna válida de contaminante
            if pollutant_col:

                # Se obtienen los valores únicos de contaminante presentes en la columna.
                # dropna() elimina valores nulos para evitar problemas.
                p_values = df[pollutant_col].dropna().unique()

                # Si existen valores válidos
                if len(p_values) > 0:

                    # Se toma el primer contaminante como referencia para el nombre.
                    base_pollutant = str(p_values[0])

                    # ------------------------------------------------------------------
                    # GENERACIÓN DEL NOMBRE BASE DEL ARCHIVO
                    # ------------------------------------------------------------------

                    # Se calcula cuántos contaminantes distintos hay en el archivo.
                    n = len(p_values)

                    # Si solo hay un contaminante:
                    # Ejemplo: ES-NO2.parquet
                    # Si hay varios contaminantes:
                    # Ejemplo: ES-NO2(3).parquet  -> indicando 3 contaminantes distintos
                    base_name = (
                        f"{country_folder.name}-{base_pollutant}"
                        if n == 1
                        else f"{country_folder.name}-{base_pollutant}({n})"
                    )
                    
                    # ------------------------------------------------------------------
                    # CONTROL DE DUPLICADOS DE NOMBRE
                    # ------------------------------------------------------------------

                    # Se incrementa el contador para ese nombre base.
                    name_counter[base_name] += 1

                    # Índice de aparición de ese nombre.
                    idx = name_counter[base_name]

                    # Si es el primer archivo con ese nombre → no se añade sufijo.
                    # Si ya existe uno previo → se añade _2, _3, etc.
                    #
                    # Ejemplo:
                    # ES-NO2.parquet
                    # ES-NO2_2.parquet
                    final_name = (
                        f"{base_name}.parquet"
                        if idx == 1
                        else f"{base_name}_{idx}.parquet"
                    )
                    
                    # Se construye la nueva ruta del archivo renombrado.
                    new_path = country_folder / final_name

                    # Si el archivo con ese nombre todavía no existe,
                    # se renombra el archivo original.
                    if not new_path.exists():
                        parquet_file.rename(new_path)

        # Si ocurre cualquier error (archivo corrupto, problema de lectura, etc.)
        # se ignora ese archivo y el proceso continúa con el siguiente.
        except:
            continue

Tras tener los archivos descargados en formato *parquet*, los procesamos para poder utilizarlos en procesos futuros. Por ello, llevabamos a cabo las siguientes acciones:

* **Conversión de formato**: transformamos los archivos de *parquet* a CSV.

* **Filtrado de variables**: seleccionamos únicamente las variables con información relevante: *SamplingPoint*, *Start*, *End*, *Pollutant*, *Value* y *Unit*, eliminando metadatos técnicos redundantes de la API y reduciendo el consumo de memoria.

* **Mantenimiento de Jerarquía**: el script replica automáticamente la estructura de carpetas por país, asegurando que los datos sigan organizados geográficamente tras la conversión.

In [9]:
# Definimos las rutas de origen y destino.
ruta_origen = BASE_PATH / "eea_parquet"
ruta_destino = BASE_PATH / "archivos_csv"

# Creamos la carpeta de destino si no existe.
ruta_destino.mkdir(parents=True, exist_ok=True)
print(f"📂 Carpeta de datos preparada en: {ruta_destino}")

📂 Carpeta de datos preparada en: ..\..\datasets\archivos_csv


In [ ]:
# ==============================================================================
# CONVERSIÓN: PARQUET A CSV (CON MANTENIMIENTO DE ESTRUCTURA CIUDAD-PAÍS)
# ==============================================================================

# Definimos las rutas de origen y destino.
ruta_origen = BASE_PATH / "eea_parquet"
ruta_destino = BASE_PATH / "archivos_csv"

# Creamos la carpeta de destino si no existe.
ruta_destino.mkdir(parents=True, exist_ok=True)
print(f"📂 Carpeta de datos preparada en: {ruta_destino}")

# FEATURE SELECTION
# ------------------------------------------------------------------------------
# Definimos las variables con las que nos queremos quedar.
# Esto reduce drásticamente el peso de los archivos CSV resultantes.
columnas_tfm = ['SamplingPoint', 'Start', 'End', 'Pollutant', 'Value', 'Unit']

# os.walk navegará por: eea_parquet -> ES -> Ciudad -> archivos.parquet
for raiz, carpetas, archivos in os.walk(ruta_origen):
    for nombre_archivo in archivos:
        if nombre_archivo.endswith('.parquet'):
            
            # Rutas de entrada
            ruta_completa_entrada = os.path.join(raiz, nombre_archivo)
            
            # Replicamos la estructura de subcarpetas (ES/Ciudad)
            ruta_relativa = os.path.relpath(raiz, ruta_origen)
            carpeta_final_destino = os.path.join(ruta_destino, ruta_relativa)
            
            # Creamos la jerarquía completa en el destino si no existe
            if not os.path.exists(carpeta_final_destino):
                os.makedirs(carpeta_final_destino)
            
            # Definimos ruta de salida
            nombre_csv = nombre_archivo.replace('.parquet', '.csv')
            ruta_completa_salida = os.path.join(carpeta_final_destino, nombre_csv)
            
            try:
                # Cargamos el archivo parquet
                df = pd.read_parquet(ruta_completa_entrada)

                # Aplicamos el filtro de columnas (solo si existen en el archivo)
                # Esto evita errores si algún archivo tiene nombres de columna distintos
                columnas_presentes = [col for col in columnas_tfm if col in df.columns]
                df_filtrado = df[columnas_presentes]

                # Guardamos en CSV
                # Usamos encoding 'utf-8-sig' por si hay tildes en nombres de ciudades
                df_filtrado.to_csv(ruta_completa_salida, index=False, encoding='utf-8-sig')
                
            except Exception as e:
                print(f"Error procesando {nombre_archivo} en {ruta_relativa}: {e}")

print("\n¡Conversión y filtrado completados!")
print(f"Los archivos CSV están listos en: {ruta_destino}")

📂 Carpeta de datos preparada en: ..\..\datasets\archivos_csv

¡Conversión y filtrado completados!
Los archivos CSV están listos en: ..\..\datasets\archivos_csv


Para finalizar el pre-procesamiento de estos archivos, unificamos los datos a nivel de contaminante. Es decir, para una misma ciudad, hay varios archivos sobre el mismo contaminante (*pollutant*), y eso se debe a que se registra el mismo contaminante a la misma hora en diferentes puntos de una misma ciudad. Por lo tanto, unificamos todos esos archivos para un mismo contaminante en una misma ciudad, siguiendo los siguientes pasos:

* **Tratamiento de multi-estaciones**: en casos donde existen múltiples puntos de muestreo sobre un mismo contaminante para una misma ciudad, aplicamos una agregación mediante la media aritmética por cada marca de tiempo (*Start*). Esto genera una serie temporal única y representativa por contaminante.

* **Optimización del *workspace***: limpiamos el almacenamiento eliminando los archivos fragmentados originales y conservando únicamente los archivos consolidados (*Unificado_POLLUTANT.csv*), optimizando así el espacio en disco y la claridad para la fase de análisis.

In [ ]:
# ==============================================================================
# UNIFICACIÓN POR CIUDAD Y CONTAMINANTE
# ==============================================================================

ruta_base_csv = BASE_PATH / "archivos_csv"

for raiz, carpetas, archivos in os.walk(ruta_base_csv):
    # Seleccionamos archivos CSV que no hayan sido unificados ya
    archivos_csv = [f for f in archivos if f.endswith('.csv') and not f.startswith("Unificado_")]
    
    if not archivos_csv:
        continue

    nombre_ciudad = os.path.basename(raiz)
    datos_subcarpeta = []

    for f in archivos_csv:
        ruta_archivo = os.path.join(raiz, f)
        try:
            df_temp = pd.read_csv(ruta_archivo)
            
            if not df_temp.empty:
                # Aseguramos que la columna 'Pollutant' existe (evita errores de mayúsculas)
                df_temp.columns = [c.strip() for c in df_temp.columns]
                datos_subcarpeta.append(df_temp)
        
        except Exception as e:
            print(f"❌ Error leyendo {f}: {e}")
    
    if datos_subcarpeta:
        # Combinamos todos los datos de la ciudad
        df_total = pd.concat(datos_subcarpeta, ignore_index=True)
        
        # Identificamos contaminantes únicos
        pollutants = df_total['Pollutant'].unique()
        
        for poll in pollutants:
            df_poll = df_total[df_total['Pollutant'] == poll]
            
            # Promediamos valores por fecha de inicio
            df_agrupado = df_poll.groupby('Start').agg({
                'Value': 'mean',
                'End': 'first',
                'Pollutant': 'first',
                'Unit': 'first',
            }).reset_index()
            
            # Guardamos con el nuevo formato de nombre y separador punto y coma
            nombre_final = f"Unificado_{nombre_ciudad}_{poll}.csv"
            ruta_salida = os.path.join(raiz, nombre_final)
            
            df_agrupado.to_csv(ruta_salida, index=False, encoding='utf-8-sig')

        # Una vez creado el archivo unificado, eliminamos los archivos fuente individuales para mantener solo el dataset listo para el análisis.
        for f in archivos_csv:
            os.remove(os.path.join(raiz, f))

print("\n¡Proceso de unificación y limpieza completado!")


¡Proceso de unificación y limpieza completado!


Así, finalmente tenemos para cada ciudad un CVS por cada contaminante del que se tiene datos, y cada CSV contiene las variables:

1. *Start*: fecha de comienzo de medición.
2. *End*: fecha de fin de medición (1 hora después).
3. *Pollutant*: nombre del contaminante. Están clasificados numéricamente.
4. *Value*: el valor del contaminante.
5. *Unit*: unidad en la que está medido el contaminante.